In [1]:
# Install CatBoost
!pip install catboost -q

import pandas as pd
import numpy as np
import os
import warnings# Install CatBoost
!pip install catboost -q

import pandas as pd
import numpy as np
import os
import warnings
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Tree Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier

# Configuration
warnings.filterwarnings('ignore')
# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported and seeds set.")

2025-12-31 15:17:40.848477: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767194261.133724      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767194261.213121      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767194261.905614      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767194261.905665      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767194261.905668      17 computation_placer.cc:177] computation placer alr

Libraries imported and seeds set.


In [2]:
# Initialize file paths
train_path = ''
test_path = ''
sub_path = ''

# Automatically search for dataset files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        if 'train.csv' in filename:
            train_path = full_path
        elif 'test.csv' in filename:
            test_path = full_path
        elif 'sample_submission.csv' in filename:
            sub_path = full_path

# Load data
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(sub_path)

print(f"Data loaded. Train: {train.shape}, Test: {test.shape}")

Data loaded. Train: (700000, 26), Test: (300000, 25)


In [3]:
# 1. Separate ID and Target
test_ids = test['id']
train = train.drop('id', axis=1)
test = test.drop('id', axis=1)

target = train['diagnosed_diabetes']
train = train.drop('diagnosed_diabetes', axis=1)

# 2. Combine for consistent encoding
train_len = len(train)
combined = pd.concat([train, test], axis=0)

# 3. One-Hot Encoding
combined_encoded = pd.get_dummies(combined, drop_first=True)

# 4. Split back
X = combined_encoded.iloc[:train_len]
X_test = combined_encoded.iloc[train_len:]

# 5. Standardization (Crucial for Neural Networks)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# Convert to DataFrame
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Preprocessing complete. Data is scaled for Neural Network compatibility.")

Preprocessing complete. Data is scaled for Neural Network compatibility.


In [4]:
# K-Fold Configuration
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Storage for predictions
oof_preds = {
    'xgb': np.zeros(len(X)),
    'lgbm': np.zeros(len(X)),
    'cat': np.zeros(len(X)),
    'nn': np.zeros(len(X)) # Placeholder for Neural Network
}

test_preds = {
    'xgb': np.zeros(len(X_test)),
    'lgbm': np.zeros(len(X_test)),
    'cat': np.zeros(len(X_test)),
    'nn': np.zeros(len(X_test)) # Placeholder for Neural Network
}

print(f"Starting Tree Models Training ({N_SPLITS}-Fold)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, target)):
    print(f"Fold {fold + 1}/{N_SPLITS}")
    
    X_tr, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_tr, y_val = target.iloc[train_idx], target.iloc[val_idx]
    
    # 1. XGBoost
    xgb = XGBClassifier(
        n_estimators=2000, learning_rate=0.015, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
        random_state=42, n_jobs=-1, early_stopping_rounds=100
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
    oof_preds['xgb'][val_idx] = xgb.predict_proba(X_val)[:, 1]
    test_preds['xgb'] += xgb.predict_proba(X_test_scaled)[:, 1] / N_SPLITS

    # 2. LightGBM
    lgbm = LGBMClassifier(
        n_estimators=2000, learning_rate=0.015, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, metric='auc',
        random_state=42, n_jobs=-1, verbosity=-1
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
             callbacks=[early_stopping(100, verbose=False)])
    oof_preds['lgbm'][val_idx] = lgbm.predict_proba(X_val)[:, 1]
    test_preds['lgbm'] += lgbm.predict_proba(X_test_scaled)[:, 1] / N_SPLITS

    # 3. CatBoost
    cat = CatBoostClassifier(
        iterations=2000, learning_rate=0.015, depth=6,
        eval_metric='AUC', random_seed=42, verbose=0,
        early_stopping_rounds=100, allow_writing_files=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    oof_preds['cat'][val_idx] = cat.predict_proba(X_val)[:, 1]
    test_preds['cat'] += cat.predict_proba(X_test_scaled)[:, 1] / N_SPLITS

print("Tree models training complete.")

Starting Tree Models Training (5-Fold)...
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5
Tree models training complete.


In [5]:
# Neural Network Training Loop
print(f"Starting Neural Network Training ({N_SPLITS}-Fold)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, target)):
    print(f"NN Fold {fold + 1}/{N_SPLITS}")
    
    X_tr, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_tr, y_val = target.iloc[train_idx], target.iloc[val_idx]
    
    # Define simple MLP architecture
    model = keras.Sequential([
        layers.Input(shape=(X_tr.shape[1],)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['AUC']
    )
    
    # Callbacks for training stability
    early_stopping_nn = callbacks.EarlyStopping(
        monitor='val_auc', patience=10, mode='max', restore_best_weights=True
    )
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.5, patience=5, min_lr=1e-6, mode='max'
    )
    
    # Train
    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        batch_size=256,
        epochs=100,
        callbacks=[early_stopping_nn, reduce_lr],
        verbose=0
    )
    
    # Predict
    oof_preds['nn'][val_idx] = model.predict(X_val, verbose=0).flatten()
    test_preds['nn'] += model.predict(X_test_scaled, verbose=0).flatten() / N_SPLITS

print("Neural Network training complete.")

Starting Neural Network Training (5-Fold)...
NN Fold 1/5


2025-12-31 15:43:07.903770: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


NN Fold 2/5
NN Fold 3/5
NN Fold 4/5
NN Fold 5/5
Neural Network training complete.


In [6]:
# Convert dictionary to DataFrame for optimization
oof_df = pd.DataFrame(oof_preds)
test_df = pd.DataFrame(test_preds)

# Define optimization function (Maximize AUC = Minimize Negative AUC)
def minimize_auc(weights):
    weights = np.array(weights)
    # Avoid division by zero
    if weights.sum() == 0: return 0
    weights /= weights.sum()
    
    # Weighted average of predictions
    final_oof = (oof_df['xgb'] * weights[0]) + \
                (oof_df['lgbm'] * weights[1]) + \
                (oof_df['cat'] * weights[2]) + \
                (oof_df['nn'] * weights[3])
    
    return -roc_auc_score(target, final_oof)

# Run optimization
print("Optimizing weights for 4 models...")
initial_weights = [0.25, 0.25, 0.25, 0.25]
result = minimize(minimize_auc, initial_weights, method='Nelder-Mead')

# Normalize weights
best_weights = result.x / result.x.sum()

print("-" * 30)
print(f"Optimal Weights:")
print(f"  XGBoost:      {best_weights[0]:.4f}")
print(f"  LightGBM:     {best_weights[1]:.4f}")
print(f"  CatBoost:     {best_weights[2]:.4f}")
print(f"  Neural Net:   {best_weights[3]:.4f}")
print(f"Optimized CV Score: {-result.fun:.5f}")
print("-" * 30)

# Create final prediction using optimized weights
final_pred = (test_df['xgb'] * best_weights[0]) + \
             (test_df['lgbm'] * best_weights[1]) + \
             (test_df['cat'] * best_weights[2]) + \
             (test_df['nn'] * best_weights[3])

# Save submission
submission_df = pd.DataFrame({
    'id': test_ids,
    'diagnosed_diabetes': final_pred
})

submission_df.to_csv('submission.csv', index=False)
print("submission.csv(v6) created successfully.")
display(submission_df.head())

Optimizing weights for 4 models...
------------------------------
Optimal Weights:
  XGBoost:      0.5662
  LightGBM:     0.9013
  CatBoost:     -0.2626
  Neural Net:   -0.2049
Optimized CV Score: 0.72724
------------------------------
submission.csv(v6) created successfully.


,id,diagnosed_diabetes
0,700000,0.487961
1,700001,0.698368
2,700002,0.796878
3,700003,0.352617
4,700004,0.935388
